<a href="https://colab.research.google.com/github/saisohanallam/ML_LAB_2026-27/blob/main/mini%20project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Profile Classification using K-Nearest Neighbors (KNN)

**Machine Learning Mini Project**

This project adapts the customer-profile theme of the reference notebook. Instead of grouping customers with K-Means (unsupervised learning), we use **K-Nearest Neighbors (KNN)** to predict a customer's profile from their age, annual income, spending score, and monthly store visits.

The dataset is generated in this notebook, so no external file upload is needed.

## 1. Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.decomposition import PCA

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Create the customer dataset

We generate 500 example customers belonging to five predefined customer profiles. Each row has four numerical features and a profile label. The labels represent the known profile used for supervised training.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

def make_group(n, age, income, spend, visits, profile):
    return pd.DataFrame({
        "Age": rng.normal(age, 4, n),
        "AnnualIncome_k": rng.normal(income, 6, n),
        "SpendingScore": rng.normal(spend, 7, n),
        "VisitsPerMonth": rng.normal(visits, 1.5, n),
        "CustomerProfile": profile
    })

df = pd.concat([
    make_group(100, 24, 30, 80, 10, "Young High Spenders"),
    make_group(100, 45, 85, 75, 8, "Affluent High Spenders"),
    make_group(100, 50, 80, 20, 2, "Affluent Low Spenders"),
    make_group(100, 55, 28, 25, 3, "Budget Conscious"),
    make_group(100, 33, 55, 50, 5, "Average Customers")
], ignore_index=True)

# Keep generated feature values within realistic positive ranges
feature_cols = ["Age", "AnnualIncome_k", "SpendingScore", "VisitsPerMonth"]
df[feature_cols] = df[feature_cols].clip(lower=1)
df[["Age", "AnnualIncome_k", "SpendingScore", "VisitsPerMonth"]] = (
    df[feature_cols].round(2)
)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Explore the dataset

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nSummary statistics:")
display(df[feature_cols].describe().round(2))

print("\nCustomers in each profile:")
display(df["CustomerProfile"].value_counts())

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="CustomerProfile", order=df["CustomerProfile"].value_counts().index)
plt.title("Number of Customers in Each Profile")
plt.xlabel("Customer Profile")
plt.ylabel("Count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 4. Prepare features and target

KNN is distance-based. We split the data first, then fit `StandardScaler` only on the training data to avoid data leakage.

In [ ]:
X = df[feature_cols]
y = df["CustomerProfile"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 5. Find a suitable value of K

We test several values of K and compare their test accuracy. The value with the highest accuracy is selected for this demonstration.

In [ ]:
k_values = range(1, 26)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    accuracies.append(accuracy_score(y_test, predictions))

best_k = list(k_values)[int(np.argmax(accuracies))]
print(f"Best K from this test split: {best_k}")
print(f"Test accuracy at best K: {max(accuracies):.3f}")

plt.figure(figsize=(8, 4))
plt.plot(list(k_values), accuracies, marker="o")
plt.title("KNN Accuracy for Different Values of K")
plt.xlabel("Number of Neighbors (K)")
plt.ylabel("Test Accuracy")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

## 6. Train the final KNN model

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric="minkowski", p=2)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"Final model accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 7. Confusion matrix

The confusion matrix compares actual customer profiles with the profiles predicted by KNN.

In [ ]:
labels = knn.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(9, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted Profile")
plt.ylabel("Actual Profile")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Visualize customers in two dimensions using PCA

PCA is used only to create a two-dimensional visualization. The KNN model itself is trained using all four scaled features.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_test_pca = pca.fit_transform(X_test_scaled)

plot_df = pd.DataFrame({
    "PC1": X_test_pca[:, 0],
    "PC2": X_test_pca[:, 1],
    "Actual Profile": y_test.to_numpy(),
    "Predicted Profile": y_pred
})

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=plot_df, x="PC1", y="PC2",
    hue="Actual Profile", style="Predicted Profile",
    s=75
)
plt.title("Customer Profiles: Actual Labels and KNN Predictions (PCA)")
plt.tight_layout()
plt.show()

print("Explained variance by two PCA components:",
      round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

## 9. Predict the profile of a new customer

Enter a customer's age, annual income (in thousands), spending score (0–100), and average store visits per month. The trained scaler and KNN model will predict the most likely profile.

In [ ]:
# Example new customer: change these values to try other customers
new_customer = pd.DataFrame([{
    "Age": 29,
    "AnnualIncome_k": 60,
    "SpendingScore": 72,
    "VisitsPerMonth": 7
}])

new_customer_scaled = scaler.transform(new_customer[feature_cols])
predicted_profile = knn.predict(new_customer_scaled)[0]
neighbor_distances, neighbor_indices = knn.kneighbors(new_customer_scaled)

print("New customer details:")
display(new_customer)
print("Predicted customer profile:", predicted_profile)
print(f"Nearest-neighbor distances: {np.round(neighbor_distances[0], 3)}")

## 10. Conclusion

- K-Nearest Neighbors is a supervised machine-learning algorithm used for classification.
- It predicts a class by looking at the labels of the nearest training examples.
- Feature scaling is important because KNN uses distances.
- The value of K affects the model's predictions; this notebook compares several values using a held-out test set.
- The final accuracy and classification report describe performance on this generated example dataset.

**Note:** This is an educational demonstration using synthetic data. Its accuracy does not represent performance on real customer records.

## 11. How KNN works

1. Choose the number of neighbors, K.
2. Calculate the distance between the new data point and training data points.
3. Select the K closest points.
4. Count their class labels.
5. Assign the most common class to the new point.

In this project, Euclidean distance is used through `metric="minkowski", p=2`.